# Laboratorio 5: Scrapling Stealth & Antibots

En el entorno real del web scraping, la mayoría de los sitios comerciales implementan medidas de seguridad que bloquean accesos automatizados. Estos sistemas detectan rápidamente cuando la conexión proviene de un script y no de un navegador humano, devolviendo típicamente un **Error 403 Forbidden** (Prohibido) o pantallas de validación de servicios como Cloudflare.

La página de reseñas literarias de Bill Gates (Gates Notes) ilustra claramente este comportamiento. En este laboratorio comprobaremos las limitaciones de las librerías básicas (`requests`) ante estas protecciones, e implementaremos **Scrapling** para emular un navegador invisible (*headless*) capaz de evadir el *firewall* y extraer la información requerida.

## Preparando el entorno

Para que un navegador invisible (*headless*) funcione, necesitamos:
1. Instalar Scrapling con **todas** sus dependencias.
2. Ejecutar `scrapling install --force` para que se descarguen los navegadores y las dependencias de sistema.

In [ ]:
# Paso 1: Instalamos Scrapling con todas sus dependencias
!pip install "scrapling[all]>=0.4.2" -q

In [ ]:
# Paso 2: Descargamos los navegadores y dependencias del sistema operativo
!scrapling install --force

## El intento ingenuo (Requests clásico)
Analizaremos el comportamiento del servidor al intentar realizar una petición a la página de Gates Notes de la misma forma que lo hicimos con el Proyecto Gutenberg en el Laboratorio 1.

In [ ]:
import requests

url = "https://www.gatesnotes.com/books/all-book-reviews"

respuesta_basica = requests.get(url)

print("Intentando entrar con Requests...")
print("Código de respuesta del servidor:", respuesta_basica.status_code)

if respuesta_basica.status_code != 200:
    print(f"¡Rebotamos! El servidor nos devolvió un código {respuesta_basica.status_code}.")
    print("Esto significa que el firewall de aplicación (WAF) bloqueó a Python.")
else:
    print("Entramos (poco común en sitios protegidos).")

## La Solución Moderna: Scrapling desde la línea de comandos

Scrapling incluye un poderoso comando llamado `scrapling extract stealthy-fetch`. Este comando opera **fuera** de Jupyter, directamente desde la terminal del sistema operativo. Este inicializa un navegador invisible equipado con técnicas anti-detección, procesa (renderiza) la página completa y guarda el resultado en un archivo local.

Esta estrategia resulta más robusta al evitar los conflictos de asincronismo (`asyncio`) frecuentes entre Jupyter y los navegadores automatizados en Windows. El flujo de trabajo consiste en delegar la extracción y renderizado a Scrapling mediante la terminal, para posteriormente analizar el resultado utilizando su *parser* nativo en Python.

In [ ]:
# Lanzamos el navegador stealth desde la línea de comandos.
# El signo '!' ejecuta el comando en la terminal del sistema, no en Python.
# Usamos --network-idle para esperar que SPA Next.js termine de cargar
# Usamos --wait 5000 para forzar una espera de 5 segundos extra.
!scrapling extract stealthy-fetch "https://www.gatesnotes.com/books/all-book-reviews" gatesnotes.html --network-idle --wait 5000

## Cargando y analizando el HTML capturado

Con el archivo `gatesnotes.html` guardado en el disco, procedemos a analizarlo empleando el *parser* nativo de Scrapling: `Selector`. Esta herramienta proporciona funcionalidades de extracción equivalentes a Beautiful Soup (`.css()`, `.xpath()`, `.find_all()`), optimizando el proceso sin requerir dependencias adicionales.

In [ ]:
from scrapling.parser import Selector

# Leemos el archivo HTML que guardó el comando anterior
with open('gatesnotes.html', 'r', encoding='utf-8') as f:
    html_crudo = f.read()

# Creamos el objeto Selector (equivalente a BeautifulSoup pero de Scrapling)
pagina = Selector(html_crudo)

print(f"HTML cargado exitosamente. Longitud: {len(html_crudo)} caracteres.")

In [ ]:
# Confirmamos que cargamos el sitio real extrayendo el título
titulo = pagina.css('title::text').get()
print("Título de la página:", titulo)

In [ ]:
# Extraemos todos los enlaces de la página para explorar la estructura
enlaces = pagina.css('a::attr(href)').getall()

print(f"Se encontraron {len(enlaces)} enlaces en la página.")
print("\nPrimeros 10 enlaces:")
for i, enlace in enumerate(enlaces[:10]):
    print(f"  {i+1}. {enlace}")

In [ ]:
# Extraemos los textos visibles de los encabezados para buscar los títulos de libros
encabezados = pagina.css('h2::text, h3::text').getall()

print(f"Se encontraron {len(encabezados)} encabezados en la página:")
for enc in encabezados:
    texto = enc.strip()
    if texto:
        print(f"  - {texto}")

Scrapling también soporta sintaxis familiar tipo Beautiful Soup (`find_all`):

In [ ]:
# Buscar elementos usando la sintaxis estilo bs4 que ya conocemos
todos_los_links = pagina.find_all('a')
print(f"Links encontrados con find_all: {len(todos_los_links)}")

## Desafío Práctico

Hasta este punto, hemos logrado ingresar a un sitio protegido y extraer elementos generales. El desafío práctico es el siguiente:

1. Abrí la página de Gates Notes en tu navegador web.
2. Hacé clic derecho sobre el título de una reseña y seleccioná **Inspeccionar Elemento**.
3. Identificá la clase CSS o la estructura HTML que envuelve cada tarjeta de libro.
4. Escribí el selector CSS correcto en la celda de abajo para extraer todos los títulos de las reseñas.

In [ ]:
# EJERCICIO: Reemplazá el selector de ejemplo por el que descubras con Inspeccionar Elemento
selector_css = "tu_selector_css_aqui"

resultados = pagina.css(selector_css).getall()

if resultados:
    print(f"¡Encontraste {len(resultados)} elementos!")
    for r in resultados[:5]:
        print(f"  - {r}")
else:
    print("El selector no devolvió resultados. Revisá la estructura del sitio con Inspeccionar Elemento.")

In [ ]:
# Limpieza: eliminamos el archivo temporal HTML
import os
if os.path.exists('gatesnotes.html'):
    os.remove('gatesnotes.html')
    print("Archivo temporal eliminado.")

---
## Desafío Práctico: Lazy Loading y React Virtualization

Al analizar nuestro archivo `gatesnotes.html`, notamos algo extraño: solo pudimos extraer los primeros **10 libros**.
En la web de Gates Notes hay 177 libros. ¿Por qué el HTML guardado no los tiene todos?

GatesNotes utiliza **Carga Diferida (Lazy Loading)** infinita, pero con una particularidad: usan un framework moderno de React (Next.js) que implementa *Virtual Scrolling*. 
Esto implica que, a medida que te desplazás por la página (scroll), **los artículos previos son removidos del código HTML (DOM)** para optimizar el uso de memoria del navegador, manteniendo renderizados únicamente los elementos visibles en pantalla.

> [!WARNING]
> Guardar el HTML al final del desplazamiento no es efectivo, ya que los primeros elementos fueron eliminados del código fuente.

### La Solución Dinámica

Para resolver esto, implementaremos **Playwright** (la tecnología fundamental de Scrapling). Al tratarse de un *framework* de automatización de menor nivel, desarrollaremos un script que:
1. Abra un navegador de incógnito real (`headed`).
2. Busque los libros que hay en pantalla.
3. Extraiga sus datos a la memoria.
4. Presione la tecla `Av Pág` para bajar.
5. Repita el proceso agregando solo los libros únicos (filtrados por URL) a una lista hasta recolectar los 177.
6. Guarde todo automáticamente en un JSON.

En la siguiente celda usaremos el comando `%%writefile` mágico de Jupyter para escribir un archivo `.py` que luego ejecutaremos por fuera del entorno.

In [ ]:
%%writefile fetch_gatesnotes_pw.py
import json
from playwright.sync_api import sync_playwright

url = "https://www.gatesnotes.com/books/all-book-reviews"

def run():
    with sync_playwright() as p:
        # Modo headed=False abre la ventana. Interesantemente, Cloudflare suele
        # bloquear menos un navegador que tiene cabeza visible.
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        print("Navegando a GatesNotes...")
        page.goto(url, wait_until="domcontentloaded")
        page.wait_for_timeout(5000)
        
        all_books = []
        seen = set() # Evita duplicados
        last_height = page.evaluate("document.body.scrollHeight")
        
        print("Extrayendo datos dinámicamente durante el desplazamiento...")
        for i in range(25):
            # Extraemos los artículos que existen AHORA MISMO en el DOM
            items = page.locator(".ArticleListItem").all()
            for item in items:
                title_el = item.locator(".CampaignArticlesTitle")
                if title_el.count() > 0:
                    title = title_el.inner_text().strip()
                    url_href = title_el.get_attribute("href")
                    
                    author_el = item.locator(".CampaignArticlesName")
                    author = author_el.inner_text().strip() if author_el.count() > 0 else ""
                    
                    cat_el = item.locator(".KBreadCrumbCopy")
                    cat = cat_el.inner_text().strip() if cat_el.count() > 0 else ""
                    
                    if url_href not in seen:
                        seen.add(url_href)
                        all_books.append({
                            "title": title, "author": author, "category": cat, "url": url_href
                        })
                        
            print(f"Iteración {i+1}: Van {len(all_books)} libros únicos extraídos.")
            
            # Simulamos la tecla "Avance de Página" para activar la carga diferida
            for _ in range(3):
                page.keyboard.press("PageDown")
                page.wait_for_timeout(500)
            
            new_height = page.evaluate("document.body.scrollHeight")
            if new_height == last_height and i > 5:
                # Comprobamos si tocamos el final exacto
                page.keyboard.press("End")
                page.wait_for_timeout(1500)
                if page.evaluate("document.body.scrollHeight") == last_height:
                    print("Llegamos al final del contenido.")
                    break
            last_height = new_height
                
        print(f"¡Extracción finalizada! Libros extraídos en total: {len(all_books)}")
        
        # Guardamos a formato JSON estructurado
        with open("gatesnotes_books.json", "w", encoding="utf-8") as f:
            json.dump(all_books, f, indent=4, ensure_ascii=False)
        print("Datos guardados en gatesnotes_books.json")
        
        browser.close()

if __name__ == "__main__":
    run()


In [ ]:
# Ahora ejecutamos el script que acabamos de guardar
# Nota: La ejecución abrirá una ventana de Chromium real en tu computadora
!python fetch_gatesnotes_pw.py

## Análisis de Datos final con Pandas

Hemos conseguido extraer un archivo JSON estructurado con títulos, autores y categorías, listo para su análisis.

In [ ]:
import pandas as pd
import json

# Cargamos el archivo JSON en un DataFrame de pandas
with open('gatesnotes_books.json', 'r', encoding='utf-8') as f:
    datos_libros = json.load(f)

df = pd.DataFrame(datos_libros)

print(f"Total de libros recuperados: {len(df)}")
display(df.head())

In [ ]:
# Analizaremos las categorías más frecuentes en las lecturas de Bill Gates
conteos_categoria = df['category'].value_counts()

# Visualizamos los 10 primeros resultados
display(conteos_categoria.head(10))

---
# Consolidación y Repaso

## Glosario Acotado
*   **Error 403 Forbidden:** Código de estado HTTP donde el servidor entiende la petición, pero se rehusa a otorgar autorización. En scraping moderno indica una barrera de Web Application Firewall (WAF).
*   **Navegador Invisible (Headless Browser):** Navegador web que opera sin interfaz gráfica de usuario. Permite la ejecución de JavaScript y el renderizado completo de páginas web para su posterior inspección de manera programática.
*   **CLI (Command Line Interface):** Interfaz de línea de comandos. En el contexto de Scrapling, posibilita la ejecución de procesos de extracción desde la terminal, mitigando conflictos con entornos interactivos como Jupyter.
*   **Selector:** Herramienta nativa de Scrapling para el análisis (*parsing*) de documentos HTML. Facilita la navegación y extracción de datos mediante métodos como `.css()`, `.xpath()` y `.find_all()`.
*   **Scrapling:** Entorno de trabajo (*framework*) moderno para recolección de datos web (*web scraping*) que combina capacidades de evasión (*stealth*), interfaz de comandos y sistemas de selección integrados.

## Preguntas de Autoevaluación

**1. ¿Por qué utilizamos el comando de terminal `scrapling extract stealthy-fetch` en lugar de instanciar el navegador directamente desde Python?**
Entornos como Jupyter implementan su propio bucle de eventos asíncronos (`asyncio`). Herramientas de automatización como Patchright requieren crear subprocesos a nivel del sistema operativo, operación que el bucle de Jupyter en Windows no soporta de forma nativa. Su ejecución mediante la terminal aísla el proceso, evitando conflictos inherentes al entorno.

**2. ¿Por qué es necesaria la ejecución de `scrapling install --force` de forma adicional a `pip install`?**
La instalación mediante `pip` únicamente adquiere las librerías de Python. Una emulación efectiva requiere binarios del navegador (como Chromium) y dependencias a nivel de sistema operativo. El comando adicional facilita la descarga e instalación automática de estos requerimientos.

**3. ¿Cuáles son las ventajas técnicas de utilizar el objeto `Selector` nativo de Scrapling respecto a Beautiful Soup?**
El objeto `Selector` centraliza múltiples paradigmas de extracción, permitiendo el uso de selectores CSS avanzados (ausentes en la implementación nativa de Beautiful Soup), expresiones XPath, métodos clásicos como `find_all()`, y heurísticas funcionales como `find_similar()`, optimizando el flujo de análisis.